# Import the rsm3d module and set the data input/oupt directories

In [1]:
from rsm3d.data_io import RSMDataLoader
from rsm3d.rsm3d import RSMBuilder  # <-- 4-circle xrayutilities builder
from rsm3d.data_viz import RSMNapariViewer
spec_file = '/Users/xiaogangyang/BNL.GOV Dropbox/Xiaogang Yang/isr_rsm3d/setup_6oct23'
setup_file = './exp_setup.yaml'
tiff_dir  = '/Users/xiaogangyang/BNL.GOV Dropbox/Xiaogang Yang/isr_rsm3d/data_6oct23_tiff'
tiff_output = '/Users/xiaogangyang/BNL.GOV Dropbox/Xiaogang Yang/isr_rsm3d/data_6oct23_tiff_cleaned'  
out_vtr   = '/Users/xiaogangyang/BNL.GOV Dropbox/Xiaogang Yang/isr_rsm3d/rsm_hkl.vtr'    # output file
scan_list = (21,)  # any list/tuple of scan numbers

# Load the data and call the RSMBuilder and compute the Q-sample and HKL for the listed frames

In [2]:
loader = RSMDataLoader(
    spec_file,
    setup_file,
    tiff_dir,
    selected_scans=(21,),
    process_hklscan_only=True,
)

builder = RSMBuilder(loader, ub_includes_2pi=True)
Q_samp, hkl, intensity = builder.compute_full()

Initialized QConversion area with:
  Sample Axis: ['x+', 'y+', 'z-']
  Detector Axis: ['x+']
  Beam Direction: (0, 1, 0)
  Wavelength: 1.080943 Å
  Distance: 0.781050 m
  Pixel Width: 0.000075 m


# Mapping the intensity with the HKL/Q_samp for 3D visualization

In [3]:
# Optional cropping
# builder.crop_by_positions(y_bound=(220, 510), x_bound=(380, 610))
# now builder.hkl, builder.Q_samp, builder.intensity are cropped

grid, (xax, yax, zax) = builder.regrid_xu(
   space="hkl",
   grid_shape=(200, None, None),
   fuzzy=True,
   normalize="mean",
   stream=True
)

# Call the napari for the 3D visualization of the RSM map

In [4]:
viz = RSMNapariViewer(
    grid, (xax, yax, zax),
    space="hkl",               # or "q"
    name="RSM",
    log_view=True,
    contrast_percentiles=(1, 99.8),
    cmap="inferno",
    rendering="attenuated_mip",  # or "mip", "translucent"
)
# launch returns the raw napari.Viewer
viewer = viz.launch()

In [5]:
# Optional cropping
builder.crop_by_positions(y_bound=(260, 510), x_bound=(380, 610))
# now builder.hkl, builder.Q_samp, builder.intensity are cropped

grid, (xax, yax, zax) = builder.regrid_xu(
   space="hkl",
   grid_shape=(200, None, None),
   fuzzy=True,
   normalize="mean",
   stream=True
)
viz = RSMNapariViewer(
    grid, (xax, yax, zax-0.06558),
    space="hkl",               # or "q"
    name="RSM",
    log_view=True,
    contrast_percentiles=(1, 99.8),
    cmap="inferno",
    rendering="attenuated_mip",  # or "mip", "translucent"
)
# launch returns the raw napari.Viewer
viewer = viz.launch()

In [9]:
from rsm3d.data_io import write_rsm_volume_to_vtr, write_rsm_volume_to_vtk
rsm = grid
edges = (xax, yax, zax)
filename = out_vtr
write_rsm_volume_to_vtk(rsm, edges, filename.replace('.vtr', '.vtk'))
write_rsm_volume_to_vtr(rsm, edges, filename, binary=False, compress=True)
